# Classificação de Imagens por CNN

Este notebook foi organizado em seções separadas por arquitetura de CNN para facilitar a execução no Google Colab.

Modelos incluídos:

- DenseNet121: baseline do artigo original
- ResNet50: CNN clássica e muito citada
- ConvNeXt V2: CNN moderna


A ideia é manter a base comum no início e deixar cada modelo em um bloco próprio.

In [1]:
# Base comum para todos os experimentos

import os
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Download da base
path = kagglehub.dataset_download("mahyeks/almond-varieties")
dataset_dir = os.path.join(path, "dataset")
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

classes = sorted(
    nome for nome in os.listdir(dataset_dir)
    if os.path.isdir(os.path.join(dataset_dir, nome))
)

image_paths = []
labels = []
for classe in classes:
    classe_dir = os.path.join(dataset_dir, classe)
    for nome_arquivo in sorted(os.listdir(classe_dir)):
        if nome_arquivo.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(classe_dir, nome_arquivo))
            labels.append(classe)

base_df = pd.DataFrame({"image_path": image_paths, "label": labels})
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(base_df["label"])
X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    base_df["image_path"],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Base carregada com sucesso.")
print(base_df["label"].value_counts())



100%|██████████| 96.5M/96.5M [00:00<00:00, 138MB/s]

Extracting files...


Base carregada com sucesso.
label
KAPADOKYA    465
AK           401
SIRA         384
NURLU        306
Name: count, dtype: int64


## DenseNet121

Baseline do artigo original. Use este bloco para a primeira comparação.

In [ ]:
# DenseNet121 - classificação com fine-tuning leve

from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np
import json
import pickle
import os
from datetime import datetime


# Limpar memória
import gc
from tensorflow.keras import backend as K

# Libera memória do modelo anterior
try:
    del model
    K.clear_session()
    gc.collect()
    print("✓ Memória do modelo anterior liberada")
except:
    pass

def salvar_resultados(nome_modelo, metricas, y_true, y_pred, y_pred_proba,
                      history=None, diretorio="resultados"):
    """
    Salva todas as métricas e resultados do modelo em arquivos locais.

    Args:
        nome_modelo: Nome identificador do modelo
        metricas: Dicionário com as métricas calculadas
        y_true: Labels verdadeiros
        y_pred: Labels preditos
        y_pred_proba: Probabilidades das predições
        history: Histórico de treinamento (opcional)
        diretorio: Pasta para salvar os resultados
    """
    # Criar diretório com timestamp para evitar sobrescrever
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pasta_modelo = os.path.join(diretorio, f"{nome_modelo}_{timestamp}")
    os.makedirs(pasta_modelo, exist_ok=True)

    # 1. Salvar métricas como JSON
    metricas_serializable = {}
    for k, v in metricas.items():
        if v is not None:
            metricas_serializable[k] = float(v) if hasattr(v, 'item') else v
        else:
            metricas_serializable[k] = None

    with open(os.path.join(pasta_modelo, "metricas.json"), 'w') as f:
        json.dump({
            "modelo": nome_modelo,
            "data_execucao": timestamp,
            "metricas": metricas_serializable
        }, f, indent=4)

    # 2. Salvar predições e labels verdadeiros
    np.save(os.path.join(pasta_modelo, "y_true.npy"), y_true)
    np.save(os.path.join(pasta_modelo, "y_pred.npy"), y_pred)
    np.save(os.path.join(pasta_modelo, "y_pred_proba.npy"), y_pred_proba)

    # 3. Salvar classification report
    from sklearn.metrics import classification_report
    report = classification_report(y_true, y_pred, output_dict=True)
    with open(os.path.join(pasta_modelo, "classification_report.json"), 'w') as f:
        json.dump(report, f, indent=4)

    # 4. Salvar histórico de treinamento (se disponível)
    if history is not None:
        hist_dict = {}
        for key, values in history.history.items():
            hist_dict[key] = [float(v) for v in values]

        with open(os.path.join(pasta_modelo, "training_history.json"), 'w') as f:
            json.dump(hist_dict, f, indent=4)

        # Também salvar como pickle para preservar tipos
        with open(os.path.join(pasta_modelo, "training_history.pkl"), 'wb') as f:
            pickle.dump(history.history, f)

    # 5. Salvar resumo executivo
    with open(os.path.join(pasta_modelo, "resumo.txt"), 'w', encoding='utf-8') as f:
        f.write("="*50 + "\n")
        f.write(f"RESUMO DO MODELO: {nome_modelo}\n")
        f.write("="*50 + "\n")
        f.write(f"Data: {timestamp}\n\n")
        f.write("MÉTRICAS:\n")
        f.write("-"*30 + "\n")
        for metrica, valor in metricas_serializable.items():
            if valor is not None:
                f.write(f"{metrica}: {valor:.4f}\n")
            else:
                f.write(f"{metrica}: Não disponível\n")

    print(f"\n✓ Resultados salvos em: {pasta_modelo}")
    print(f"  - metricas.json")
    print(f"  - y_true.npy e y_pred.npy")
    print(f"  - y_pred_proba.npy")
    print(f"  - classification_report.json")
    if history is not None:
        print(f"  - training_history.json/pkl")
    print(f"  - resumo.txt")

    return pasta_modelo


def calcular_metricas(y_true, y_pred, y_pred_proba=None, num_classes=None):
    """
    Calcula todas as métricas e retorna um dicionário com os resultados.
    Para AUC, é necessário passar as probabilidades (y_pred_proba).
    """
    metricas = {}

    # Métricas básicas
    metricas['Accuracy'] = accuracy_score(y_true, y_pred)
    metricas['Precision'] = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    metricas['Recall'] = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    metricas['F1-Score'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    metricas['Kappa'] = cohen_kappa_score(y_true, y_pred)

    # AUC (requer probabilidades)
    if y_pred_proba is not None:
        if num_classes is None:
            num_classes = y_pred_proba.shape[1]

        # Para classificação binária
        if num_classes == 2:
            metricas['AUC'] = roc_auc_score(y_true, y_pred_proba[:, 1])
        # Para classificação multiclasse (one-vs-rest)
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metricas['AUC'] = roc_auc_score(y_true_bin, y_pred_proba,
                                           multi_class='ovr', average='weighted')
    else:
        metricas['AUC'] = None

    return metricas


def carregar_imagens(lista_paths, target_size=(224, 224)):
    imagens = []
    for image_path in lista_paths:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        imagens.append(img_array)
    return preprocess_input(np.asarray(imagens, dtype=np.float32))


# ============================================
# EXECUÇÃO PRINCIPAL
# ============================================

X_train = carregar_imagens(X_train_paths.tolist())
X_test = carregar_imagens(X_test_paths.tolist())
os.makedirs("models", exist_ok=True)

base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model = Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treinamento
history = model.fit(
     X_train,
     y_train,
     validation_split=0.2,
     epochs=50,
     batch_size=32,
     verbose=1,
 )

# Salvar modelo treinado
model.save("models/densenet121.keras")
print("✓ Modelo salvo em: models/densenet121.keras")

# Avaliação com métricas
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# Calcula todas as métricas
metricas = calcular_metricas(y_test, y_pred, y_pred_proba, num_classes=len(label_encoder.classes_))

# Exibe as métricas
print("\n" + "="*50)
print("MÉTRICAS DO MODELO DenseNet121")
print("="*50)
for metrica, valor in metricas.items():
    if valor is not None:
        print(f"{metrica}: {valor:.4f}")
    else:
        print(f"{metrica}: Não disponível")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# ============================================
# SALVAR RESULTADOS LOCALMENTE
# ============================================
pasta_resultados = salvar_resultados(
    nome_modelo="DenseNet121",
    metricas=metricas,
    y_true=y_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba,
    history=history,
    diretorio="resultados"
)

print(f"\n✓ Todos os resultados do DenseNet121 foram salvos com sucesso!")
print(f"✓ Pasta: {pasta_resultados}")

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/50


In [ ]:
# ============================================
# VISUALIZAÇÃO DOS RESULTADOS - DenseNet121
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import json
import os
from pathlib import Path

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# ============================================
# 1. CARREGAR RESULTADOS SALVOS
# ============================================
def carregar_resultados(pasta_resultados):
    """Carrega os resultados salvos do modelo"""
    resultados = {}

    # Carregar métricas
    with open(os.path.join(pasta_resultados, "metricas.json"), 'r') as f:
        resultados['metricas'] = json.load(f)

    # Carregar arrays numpy
    resultados['y_true'] = np.load(os.path.join(pasta_resultados, "y_true.npy"))
    resultados['y_pred'] = np.load(os.path.join(pasta_resultados, "y_pred.npy"))
    resultados['y_pred_proba'] = np.load(os.path.join(pasta_resultados, "y_pred_proba.npy"))

    # Carregar classification report
    with open(os.path.join(pasta_resultados, "classification_report.json"), 'r') as f:
        resultados['classification_report'] = json.load(f)

    # Carregar histórico se existir
    history_path = os.path.join(pasta_resultados, "training_history.json")
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            resultados['history'] = json.load(f)

    return resultados

# Carregar dados
print("Carregando resultados salvos...")
resultados = carregar_resultados(pasta_resultados)
print("✓ Resultados carregados com sucesso!")

# ============================================
# 2. TABELA DE MÉTRICAS
# ============================================
print("\n" + "="*60)
print("📊 TABELA DE MÉTRICAS - DenseNet121")
print("="*60)

# Criar DataFrame com as métricas
metricas_dict = resultados['metricas']['metricas']
df_metricas = pd.DataFrame({
    'Métrica': list(metricas_dict.keys()),
    'Valor': list(metricas_dict.values())
})

# Formatar valores
def format_bar(value):
    """Cria uma barra de progresso visual"""
    if value is None or value == "N/A":
        return "Não disponível"
    try:
        val = float(value)
        bar_length = int(val * 20)  # 20 caracteres máximos
        bar = "█" * bar_length + "░" * (20 - bar_length)
        return f"{bar} {val:.4f}"
    except:
        return str(value)

df_metricas['Visualização'] = df_metricas['Valor'].apply(format_bar)

# Exibir tabela formatada
print("\nMÉTRICAS DE DESEMPENHO:")
print("-"*60)
for idx, row in df_metricas.iterrows():
    print(f"  {row['Métrica']:<12} {row['Visualização']}")
print("-"*60)

# ============================================
# 3. MATRIZ DE CONFUSÃO
# ============================================
print("\n" + "="*60)
print("🎯 MATRIZ DE CONFUSÃO - DenseNet121")
print("="*60)

# Obter classes
if hasattr(label_encoder, 'classes_'):
    nomes_classes = [str(c) for c in label_encoder.classes_]
else:
    # Fallback: deduzir classes únicas
    nomes_classes = sorted(set(str(c) for c in np.unique(np.concatenate([resultados['y_true'], resultados['y_pred']]))))

n_classes = len(nomes_classes)
print(f"Classes detectadas: {n_classes}")
print(f"Nomes: {', '.join(nomes_classes[:5])}{'...' if len(nomes_classes) > 5 else ''}")

# Calcular matriz de confusão
cm = confusion_matrix(resultados['y_true'], resultados['y_pred'])

# Criar figura com subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Para muitas classes, ajustar tamanho da fonte
if n_classes > 10:
    annot_fontsize = 8
    rotation_angle = 45
else:
    annot_fontsize = 10
    rotation_angle = 0

# ===== Matriz de Confusão - Contagens =====
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=nomes_classes,
            yticklabels=nomes_classes,
            ax=axes[0],
            annot_kws={'size': annot_fontsize},
            cbar_kws={'label': 'Contagem'})
axes[0].set_title('Matriz de Confusão - Contagens', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classe Predita', fontsize=12)
axes[0].set_ylabel('Classe Verdadeira', fontsize=12)
axes[0].tick_params(axis='x', rotation=rotation_angle)
axes[0].tick_params(axis='y', rotation=0)

# ===== Matriz de Confusão - Percentuais =====
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_percent,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            xticklabels=nomes_classes,
            yticklabels=nomes_classes,
            ax=axes[1],
            vmin=0,
            vmax=100,
            annot_kws={'size': annot_fontsize},
            cbar_kws={'label': '%'})
axes[1].set_title('Matriz de Confusão - Percentuais (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Classe Predita', fontsize=12)
axes[1].set_ylabel('Classe Verdadeira', fontsize=12)
axes[1].tick_params(axis='x', rotation=rotation_angle)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle('DenseNet121 - Análise de Classificação', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(pasta_resultados, 'matriz_confusao.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Matriz de confusão salva em: matriz_confusao.png")

# ============================================
# 4. MÉTRICAS POR CLASSE (CORRIGIDO)
# ============================================
print("\n" + "="*60)
print("📈 MÉTRICAS POR CLASSE")
print("="*60)

# Extrair métricas por classe do classification report
report = resultados['classification_report']

# Filtrar apenas as entradas numéricas (classes)
classes_no_report = [k for k in report.keys() if k not in ['accuracy', 'macro avg', 'weighted avg'] and isinstance(report[k], dict)]

if not classes_no_report:
    print("⚠️ Nenhuma classe individual encontrada no report. Usando índices numéricos...")
    # Criar mapeamento se necessário
    classes_no_report = [str(c) for c in sorted(set(resultados['y_true']))]

    # Recalcular métricas por classe se não estiverem no report
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, support = precision_recall_fscore_support(
        resultados['y_true'],
        resultados['y_pred'],
        average=None
    )

    report_por_classe = {}
    for i, classe in enumerate(classes_no_report):
        report_por_classe[classe] = {
            'precision': precision[i] if i < len(precision) else 0,
            'recall': recall[i] if i < len(recall) else 0,
            'f1-score': f1[i] if i < len(f1) else 0,
            'support': support[i] if i < len(support) else 0
        }
else:
    report_por_classe = {classe: report[classe] for classe in classes_no_report}

# Criar DataFrame com métricas por classe
df_por_classe = pd.DataFrame()
for classe, metrics in report_por_classe.items():
    df_por_classe[classe] = {
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1-score'],
        'Support': metrics['support']
    }

df_por_classe = df_por_classe.round(3)

# Adicionar médias
for avg_type in ['macro avg', 'weighted avg']:
    if avg_type in report:
        df_por_classe[avg_type] = {
            'Precision': report[avg_type]['precision'],
            'Recall': report[avg_type]['recall'],
            'F1-Score': report[avg_type]['f1-score'],
            'Support': report[avg_type]['support']
        }

print("\nTabela de métricas por classe:")
print("="*60)
# Mostrar apenas as primeiras e últimas classes se houver muitas
if len(df_por_classe.columns) > 10:
    print("(Mostrando primeiras 5 e últimas 5 classes)")
    pd.set_option('display.max_columns', 12)
print(df_por_classe.to_string())
print("="*60)

# ============================================
# 5. RESUMO VISUAL (CORRIGIDO)
# ============================================
print("\n" + "="*60)
print("🎯 RESUMO DO DESEMPENHO")
print("="*60)

# Encontrar melhor e pior classe usando o report_por_classe
if report_por_classe:
    # Ordenar classes por F1-Score
    f1_scores = {classe: metrics['f1-score'] for classe, metrics in report_por_classe.items()}
    if f1_scores:
        melhor_classe = max(f1_scores, key=f1_scores.get)
        pior_classe = min(f1_scores, key=f1_scores.get)

        print(f"✓ Acurácia geral: {metricas_dict['Accuracy']:.4f}")
        print(f"✓ Número de classes: {len(report_por_classe)}")
        print(f"✓ Melhor classe: {melhor_classe} (F1-Score: {f1_scores[melhor_classe]:.3f})")
        print(f"✓ Pior classe: {pior_classe} (F1-Score: {f1_scores[pior_classe]:.3f})")
        print(f"✓ Total de imagens de teste: {len(resultados['y_true'])}")

        # Distribuição das classes
        print(f"\nDistribuição - Top 5 classes com melhor F1-Score:")
        top5 = sorted(f1_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        for i, (classe, f1) in enumerate(top5, 1):
            print(f"  {i}. {classe}: {f1:.3f}")

        print(f"\nDistribuição - Top 5 classes com pior F1-Score:")
        bottom5 = sorted(f1_scores.items(), key=lambda x: x[1])[:5]
        for i, (classe, f1) in enumerate(bottom5, 1):
            print(f"  {i}. {classe}: {f1:.3f}")
else:
    print("⚠️ Não foi possível extrair métricas por classe.")

# Salvar tabelas como CSV
df_metricas.to_csv(os.path.join(pasta_resultados, 'tabela_metricas.csv'), index=False)
df_por_classe.to_csv(os.path.join(pasta_resultados, 'metricas_por_classe.csv'))

print(f"\n✓ Tabelas salvas em: {pasta_resultados}")
print(f"  - tabela_metricas.csv")
print(f"  - metricas_por_classe.csv")
print(f"  - matriz_confusao.png")

# ============================================
# 6. GRÁFICO DE BARRAS DAS MÉTRICAS
# ============================================
print("\n" + "="*60)
print("📊 GERANDO GRÁFICO DE BARRAS")
print("="*60)

# Filtrar apenas métricas numéricas
metricas_plot = {k: v for k, v in metricas_dict.items() if v is not None}

if metricas_plot:
    fig, ax = plt.subplots(figsize=(10, 6))

    # Criar barras com gradiente de cores
    n_barras = len(metricas_plot)
    cores = plt.cm.viridis(np.linspace(0.2, 0.9, n_barras))
    barras = ax.bar(range(n_barras),
                    list(metricas_plot.values()),
                    color=cores,
                    edgecolor='black',
                    linewidth=1.5)

    # Adicionar valores nas barras
    for barra, (nome, valor) in zip(barras, metricas_plot.items()):
        height = barra.get_height()
        ax.text(barra.get_x() + barra.get_width()/2., height + 0.01,
                f'{valor:.4f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

    # Configurar eixos
    ax.set_xticks(range(n_barras))
    ax.set_xticklabels(list(metricas_plot.keys()), rotation=45, ha='right')
    ax.set_ylabel('Valor', fontsize=12, fontweight='bold')
    ax.set_title('DenseNet121 - Métricas de Desempenho', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

    # Adicionar linhas de referência
    ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Excelente (0.8)')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Bom (0.6)')
    ax.axhline(y=0.4, color='red', linestyle='--', alpha=0.5, label='Regular (0.4)')
    ax.legend(loc='lower left')

    plt.tight_layout()
    plt.savefig(os.path.join(pasta_resultados, 'grafico_metricas.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Gráfico de métricas salvo em: grafico_metricas.png")
else:
    print("⚠️ Nenhuma métrica disponível para plotar.")

print("\n✅ Visualização completa dos resultados!")

## ResNet50

CNN clássica e muito citada. Use este bloco para o segundo experimento.

In [ ]:
# ResNet50 - CNN clássica e muito citada com salvamento de métricas

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np
import json
import pickle
import os
from datetime import datetime

# Limpar memória antes da ResNet50
import gc
from tensorflow.keras import backend as K

# Libera memória do modelo anterior (DenseNet121)
try:
    del model
    K.clear_session()
    gc.collect()
    print("✓ Memória do modelo anterior liberada")
except:
    pass


def salvar_resultados_resnet(nome_modelo, metricas, y_true, y_pred, y_pred_proba,
                             history=None, diretorio="resultados"):
    """
    Salva todas as métricas e resultados do modelo ResNet50 em arquivos locais.

    Args:
        nome_modelo: Nome identificador do modelo
        metricas: Dicionário com as métricas calculadas
        y_true: Labels verdadeiros
        y_pred: Labels preditos
        y_pred_proba: Probabilidades das predições
        history: Histórico de treinamento (opcional)
        diretorio: Pasta para salvar os resultados
    """
    # Criar diretório com timestamp para evitar sobrescrever
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pasta_modelo = os.path.join(diretorio, f"{nome_modelo}_{timestamp}")
    os.makedirs(pasta_modelo, exist_ok=True)

    # 1. Salvar métricas como JSON
    metricas_serializable = {}
    for k, v in metricas.items():
        if v is not None:
            metricas_serializable[k] = float(v) if hasattr(v, 'item') else v
        else:
            metricas_serializable[k] = None

    with open(os.path.join(pasta_modelo, "metricas.json"), 'w') as f:
        json.dump({
            "modelo": nome_modelo,
            "data_execucao": timestamp,
            "metricas": metricas_serializable
        }, f, indent=4)

    # 2. Salvar predições e labels verdadeiros
    np.save(os.path.join(pasta_modelo, "y_true.npy"), y_true)
    np.save(os.path.join(pasta_modelo, "y_pred.npy"), y_pred)
    np.save(os.path.join(pasta_modelo, "y_pred_proba.npy"), y_pred_proba)

    # 3. Salvar classification report
    from sklearn.metrics import classification_report
    report = classification_report(y_true, y_pred, output_dict=True)
    with open(os.path.join(pasta_modelo, "classification_report.json"), 'w') as f:
        json.dump(report, f, indent=4)

    # 4. Salvar histórico de treinamento (se disponível)
    if history is not None:
        hist_dict = {}
        for key, values in history.history.items():
            hist_dict[key] = [float(v) for v in values]

        with open(os.path.join(pasta_modelo, "training_history.json"), 'w') as f:
            json.dump(hist_dict, f, indent=4)

        # Também salvar como pickle para preservar tipos
        with open(os.path.join(pasta_modelo, "training_history.pkl"), 'wb') as f:
            pickle.dump(history.history, f)

    # 5. Salvar resumo executivo
    with open(os.path.join(pasta_modelo, "resumo.txt"), 'w', encoding='utf-8') as f:
        f.write("="*50 + "\n")
        f.write(f"RESUMO DO MODELO: {nome_modelo}\n")
        f.write("="*50 + "\n")
        f.write(f"Data: {timestamp}\n\n")
        f.write("MÉTRICAS:\n")
        f.write("-"*30 + "\n")
        for metrica, valor in metricas_serializable.items():
            if valor is not None:
                f.write(f"{metrica}: {valor:.4f}\n")
            else:
                f.write(f"{metrica}: Não disponível\n")

        # Adicionar informações sobre o modelo
        f.write("\nCONFIGURAÇÃO DO MODELO:\n")
        f.write("-"*30 + "\n")
        f.write(f"Arquitetura: ResNet50\n")
        f.write(f"Pesos: ImageNet (pré-treinado)\n")
        f.write(f"Fine-tuning: Camadas base congeladas\n")
        f.write(f"Otimizador: Adam (lr=1e-4)\n")
        f.write(f"Loss: Sparse Categorical Crossentropy\n")
        f.write(f"Batch Size: 32\n")
        f.write(f"Epochs: 2\n")

    print(f"\n✓ Resultados salvos em: {pasta_modelo}")
    print(f"  - metricas.json")
    print(f"  - y_true.npy e y_pred.npy")
    print(f"  - y_pred_proba.npy")
    print(f"  - classification_report.json")
    if history is not None:
        print(f"  - training_history.json/pkl")
    print(f"  - resumo.txt")

    return pasta_modelo


def calcular_metricas(y_true, y_pred, y_pred_proba=None, num_classes=None):
    """
    Calcula todas as métricas e retorna um dicionário com os resultados.
    Para AUC, é necessário passar as probabilidades (y_pred_proba).
    """
    metricas = {}

    # Métricas básicas
    metricas['Accuracy'] = accuracy_score(y_true, y_pred)
    metricas['Precision'] = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    metricas['Recall'] = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    metricas['F1-Score'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    metricas['Kappa'] = cohen_kappa_score(y_true, y_pred)

    # AUC (requer probabilidades)
    if y_pred_proba is not None:
        if num_classes is None:
            num_classes = y_pred_proba.shape[1]

        # Para classificação binária
        if num_classes == 2:
            metricas['AUC'] = roc_auc_score(y_true, y_pred_proba[:, 1])
        # Para classificação multiclasse (one-vs-rest)
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metricas['AUC'] = roc_auc_score(y_true_bin, y_pred_proba,
                                           multi_class='ovr', average='weighted')
    else:
        metricas['AUC'] = None

    return metricas


def carregar_imagens_resnet(lista_paths, target_size=(224, 224)):
    imagens = []
    for image_path in lista_paths:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        imagens.append(img_array)
    return preprocess_input(np.asarray(imagens, dtype=np.float32))


# ============================================
# EXECUÇÃO PRINCIPAL
# ============================================

print("="*60)
print("RESNET50 - CARREGAMENTO E TREINAMENTO")
print("="*60)

# Carregar dados
print("\nCarregando imagens...")
X_train_resnet = carregar_imagens_resnet(X_train_paths.tolist())
X_test_resnet = carregar_imagens_resnet(X_test_paths.tolist())
print(f"✓ X_train: {X_train_resnet.shape}")
print(f"✓ X_test: {X_test_resnet.shape}")

os.makedirs("models", exist_ok=True)

# Criar modelo
print("\nConstruindo modelo ResNet50...")
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model_resnet = Model(inputs=base_model.input, outputs=output)

# Congelar camadas base
for layer in base_model.layers:
    layer.trainable = False

print(f"✓ Total de camadas: {len(model_resnet.layers)}")
print(f"✓ Camadas treináveis: {len([l for l in model_resnet.layers if l.trainable])}")
print(f"✓ Camadas congeladas: {len([l for l in model_resnet.layers if not l.trainable])}")

model_resnet.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treinamento
print("\nIniciando treinamento...")
history = model_resnet.fit(
     X_train_resnet,
     y_train,
     validation_split=0.2,
     epochs=50,
     batch_size=32,
     verbose=1,
 )

# Salvar modelo treinado
model_resnet.save("models/resnet50.keras")
print("\n✓ Modelo salvo em: models/resnet50.keras")

# Avaliação com métricas
print("\nAvaliando modelo...")
y_pred_proba_resnet = model_resnet.predict(X_test_resnet, verbose=0)
y_pred_resnet = np.argmax(y_pred_proba_resnet, axis=1)

# Calcula todas as métricas para o ResNet50
metricas_resnet = calcular_metricas(
    y_test,
    y_pred_resnet,
    y_pred_proba_resnet,
    num_classes=len(label_encoder.classes_)
)

# Exibe as métricas
print("\n" + "="*50)
print("MÉTRICAS DO MODELO ResNet50")
print("="*50)
for metrica, valor in metricas_resnet.items():
    if valor is not None:
        print(f"  {metrica:<12}: {valor:.4f}")
    else:
        print(f"  {metrica:<12}: Não disponível")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_resnet, target_names=label_encoder.classes_))

# ============================================
# SALVAR RESULTADOS LOCALMENTE
# ============================================

print("\n" + "="*60)
print("SALVANDO RESULTADOS")
print("="*60)

pasta_resultados_resnet = salvar_resultados_resnet(
    nome_modelo="ResNet50",
    metricas=metricas_resnet,
    y_true=y_test,
    y_pred=y_pred_resnet,
    y_pred_proba=y_pred_proba_resnet,
    history=history,
    diretorio="resultados"
)

print(f"\n{'='*60}")
print(f"✅ RESUMO FINAL - ResNet50")
print(f"{'='*60}")
print(f"✓ Acurácia: {metricas_resnet['Accuracy']:.4f}")
print(f"✓ F1-Score: {metricas_resnet['F1-Score']:.4f}")
if metricas_resnet['AUC'] is not None:
    print(f"✓ AUC: {metricas_resnet['AUC']:.4f}")
print(f"✓ Kappa: {metricas_resnet['Kappa']:.4f}")
print(f"\n✓ Todos os resultados foram salvos com sucesso!")
print(f"✓ Pasta: {pasta_resultados_resnet}")
print(f"\nPróximo passo: Execute a célula de visualização para gerar gráficos e tabelas.")

In [ ]:
# ============================================
# VISUALIZAÇÃO DOS RESULTADOS - ResNet50
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import json
import os
from pathlib import Path

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# ============================================
# 1. CARREGAR RESULTADOS SALVOS
# ============================================

# ============================================
# ENCONTRAR AUTOMATICAMENTE A PASTA DA RESNET50
# ============================================

import os
import glob

def encontrar_pasta_resultados(nome_modelo="ResNet50", diretorio_base="resultados"):
    """
    Encontra automaticamente a pasta de resultados mais recente para um modelo.
    """
    # Listar todas as pastas que começam com o nome do modelo
    padrao = os.path.join(diretorio_base, f"{nome_modelo}_*")
    pastas = sorted(glob.glob(padrao))

    if not pastas:
        raise FileNotFoundError(
            f"Nenhuma pasta encontrada para {nome_modelo} em {diretorio_base}/\n"
            f"Verifique se o treinamento foi executado e os resultados foram salvos."
        )

    # Retornar a pasta mais recente (última em ordem alfabética = mais recente)
    pasta_mais_recente = pastas[-1]

    print(f"📁 Pastas encontradas para {nome_modelo}:")
    for i, pasta in enumerate(pastas, 1):
        if pasta == pasta_mais_recente:
            print(f"  {i}. {os.path.basename(pasta)} ← SELECIONADA (mais recente)")
        else:
            print(f"  {i}. {os.path.basename(pasta)}")

    return pasta_mais_recente

# Encontrar pasta automaticamente
pasta_resultados = encontrar_pasta_resultados("ResNet50")
print(f"\n✓ Usando: {pasta_resultados}")

def carregar_resultados(pasta_resultados):
    """Carrega os resultados salvos do modelo"""
    resultados = {}

    # Carregar métricas
    with open(os.path.join(pasta_resultados, "metricas.json"), 'r') as f:
        resultados['metricas'] = json.load(f)

    # Carregar arrays numpy
    resultados['y_true'] = np.load(os.path.join(pasta_resultados, "y_true.npy"))
    resultados['y_pred'] = np.load(os.path.join(pasta_resultados, "y_pred.npy"))
    resultados['y_pred_proba'] = np.load(os.path.join(pasta_resultados, "y_pred_proba.npy"))

    # Carregar classification report
    with open(os.path.join(pasta_resultados, "classification_report.json"), 'r') as f:
        resultados['classification_report'] = json.load(f)

    # Carregar histórico se existir
    history_path = os.path.join(pasta_resultados, "training_history.json")
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            resultados['history'] = json.load(f)

    return resultados

# Carregar dados
print("Carregando resultados salvos...")
resultados = carregar_resultados(pasta_resultados)
print("✓ Resultados carregados com sucesso!")

# ============================================
# 2. TABELA DE MÉTRICAS
# ============================================
print("\n" + "="*60)
print("📊 TABELA DE MÉTRICAS - ResNet50")
print("="*60)

# Criar DataFrame com as métricas
metricas_dict = resultados['metricas']['metricas']
df_metricas = pd.DataFrame({
    'Métrica': list(metricas_dict.keys()),
    'Valor': list(metricas_dict.values())
})

# Formatar valores
def format_bar(value):
    """Cria uma barra de progresso visual"""
    if value is None or value == "N/A":
        return "Não disponível"
    try:
        val = float(value)
        bar_length = int(val * 20)  # 20 caracteres máximos
        bar = "█" * bar_length + "░" * (20 - bar_length)
        return f"{bar} {val:.4f}"
    except:
        return str(value)

df_metricas['Visualização'] = df_metricas['Valor'].apply(format_bar)

# Exibir tabela formatada
print("\nMÉTRICAS DE DESEMPENHO:")
print("-"*60)
for idx, row in df_metricas.iterrows():
    print(f"  {row['Métrica']:<12} {row['Visualização']}")
print("-"*60)

# ============================================
# 3. MATRIZ DE CONFUSÃO
# ============================================
print("\n" + "="*60)
print("🎯 MATRIZ DE CONFUSÃO - ResNet50")
print("="*60)

# Obter classes
if hasattr(label_encoder, 'classes_'):
    nomes_classes = [str(c) for c in label_encoder.classes_]
else:
    # Fallback: deduzir classes únicas
    nomes_classes = sorted(set(str(c) for c in np.unique(np.concatenate([resultados['y_true'], resultados['y_pred']]))))

n_classes = len(nomes_classes)
print(f"Classes detectadas: {n_classes}")
print(f"Nomes: {', '.join(nomes_classes[:5])}{'...' if len(nomes_classes) > 5 else ''}")

# Calcular matriz de confusão
cm = confusion_matrix(resultados['y_true'], resultados['y_pred'])

# Criar figura com subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Para muitas classes, ajustar tamanho da fonte
if n_classes > 10:
    annot_fontsize = 8
    rotation_angle = 45
else:
    annot_fontsize = 10
    rotation_angle = 0

# ===== Matriz de Confusão - Contagens =====
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=nomes_classes,
            yticklabels=nomes_classes,
            ax=axes[0],
            annot_kws={'size': annot_fontsize},
            cbar_kws={'label': 'Contagem'})
axes[0].set_title('Matriz de Confusão - Contagens', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classe Predita', fontsize=12)
axes[0].set_ylabel('Classe Verdadeira', fontsize=12)
axes[0].tick_params(axis='x', rotation=rotation_angle)
axes[0].tick_params(axis='y', rotation=0)

# ===== Matriz de Confusão - Percentuais =====
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_percent,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            xticklabels=nomes_classes,
            yticklabels=nomes_classes,
            ax=axes[1],
            vmin=0,
            vmax=100,
            annot_kws={'size': annot_fontsize},
            cbar_kws={'label': '%'})
axes[1].set_title('Matriz de Confusão - Percentuais (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Classe Predita', fontsize=12)
axes[1].set_ylabel('Classe Verdadeira', fontsize=12)
axes[1].tick_params(axis='x', rotation=rotation_angle)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle('ResNet50 - Análise de Classificação', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(pasta_resultados, 'matriz_confusao.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Matriz de confusão salva em: matriz_confusao.png")

# ============================================
# 4. MÉTRICAS POR CLASSE (CORRIGIDO)
# ============================================
print("\n" + "="*60)
print("📈 MÉTRICAS POR CLASSE")
print("="*60)

# Extrair métricas por classe do classification report
report = resultados['classification_report']

# Filtrar apenas as entradas numéricas (classes)
classes_no_report = [k for k in report.keys() if k not in ['accuracy', 'macro avg', 'weighted avg'] and isinstance(report[k], dict)]

if not classes_no_report:
    print("⚠️ Nenhuma classe individual encontrada no report. Usando índices numéricos...")
    # Criar mapeamento se necessário
    classes_no_report = [str(c) for c in sorted(set(resultados['y_true']))]

    # Recalcular métricas por classe se não estiverem no report
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, support = precision_recall_fscore_support(
        resultados['y_true'],
        resultados['y_pred'],
        average=None
    )

    report_por_classe = {}
    for i, classe in enumerate(classes_no_report):
        report_por_classe[classe] = {
            'precision': precision[i] if i < len(precision) else 0,
            'recall': recall[i] if i < len(recall) else 0,
            'f1-score': f1[i] if i < len(f1) else 0,
            'support': support[i] if i < len(support) else 0
        }
else:
    report_por_classe = {classe: report[classe] for classe in classes_no_report}

# Criar DataFrame com métricas por classe
df_por_classe = pd.DataFrame()
for classe, metrics in report_por_classe.items():
    df_por_classe[classe] = {
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1-score'],
        'Support': metrics['support']
    }

df_por_classe = df_por_classe.round(3)

# Adicionar médias
for avg_type in ['macro avg', 'weighted avg']:
    if avg_type in report:
        df_por_classe[avg_type] = {
            'Precision': report[avg_type]['precision'],
            'Recall': report[avg_type]['recall'],
            'F1-Score': report[avg_type]['f1-score'],
            'Support': report[avg_type]['support']
        }

print("\nTabela de métricas por classe:")
print("="*60)
# Mostrar apenas as primeiras e últimas classes se houver muitas
if len(df_por_classe.columns) > 10:
    print("(Mostrando primeiras 5 e últimas 5 classes)")
    pd.set_option('display.max_columns', 12)
print(df_por_classe.to_string())
print("="*60)

# ============================================
# 5. RESUMO VISUAL (CORRIGIDO)
# ============================================
print("\n" + "="*60)
print("🎯 RESUMO DO DESEMPENHO")
print("="*60)

# Encontrar melhor e pior classe usando o report_por_classe
if report_por_classe:
    # Ordenar classes por F1-Score
    f1_scores = {classe: metrics['f1-score'] for classe, metrics in report_por_classe.items()}
    if f1_scores:
        melhor_classe = max(f1_scores, key=f1_scores.get)
        pior_classe = min(f1_scores, key=f1_scores.get)

        print(f"✓ Acurácia geral: {metricas_dict['Accuracy']:.4f}")
        print(f"✓ Número de classes: {len(report_por_classe)}")
        print(f"✓ Melhor classe: {melhor_classe} (F1-Score: {f1_scores[melhor_classe]:.3f})")
        print(f"✓ Pior classe: {pior_classe} (F1-Score: {f1_scores[pior_classe]:.3f})")
        print(f"✓ Total de imagens de teste: {len(resultados['y_true'])}")

        # Distribuição das classes
        print(f"\nDistribuição - Top 5 classes com melhor F1-Score:")
        top5 = sorted(f1_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        for i, (classe, f1) in enumerate(top5, 1):
            print(f"  {i}. {classe}: {f1:.3f}")

        print(f"\nDistribuição - Top 5 classes com pior F1-Score:")
        bottom5 = sorted(f1_scores.items(), key=lambda x: x[1])[:5]
        for i, (classe, f1) in enumerate(bottom5, 1):
            print(f"  {i}. {classe}: {f1:.3f}")
else:
    print("⚠️ Não foi possível extrair métricas por classe.")

# Salvar tabelas como CSV
df_metricas.to_csv(os.path.join(pasta_resultados, 'tabela_metricas.csv'), index=False)
df_por_classe.to_csv(os.path.join(pasta_resultados, 'metricas_por_classe.csv'))

print(f"\n✓ Tabelas salvas em: {pasta_resultados}")
print(f"  - tabela_metricas.csv")
print(f"  - metricas_por_classe.csv")
print(f"  - matriz_confusao.png")

# ============================================
# 6. GRÁFICO DE BARRAS DAS MÉTRICAS
# ============================================
print("\n" + "="*60)
print("📊 GERANDO GRÁFICO DE BARRAS")
print("="*60)

# Filtrar apenas métricas numéricas
metricas_plot = {k: v for k, v in metricas_dict.items() if v is not None}

if metricas_plot:
    fig, ax = plt.subplots(figsize=(10, 6))

    # Criar barras com gradiente de cores
    n_barras = len(metricas_plot)
    cores = plt.cm.viridis(np.linspace(0.2, 0.9, n_barras))
    barras = ax.bar(range(n_barras),
                    list(metricas_plot.values()),
                    color=cores,
                    edgecolor='black',
                    linewidth=1.5)

    # Adicionar valores nas barras
    for barra, (nome, valor) in zip(barras, metricas_plot.items()):
        height = barra.get_height()
        ax.text(barra.get_x() + barra.get_width()/2., height + 0.01,
                f'{valor:.4f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

    # Configurar eixos
    ax.set_xticks(range(n_barras))
    ax.set_xticklabels(list(metricas_plot.keys()), rotation=45, ha='right')
    ax.set_ylabel('Valor', fontsize=12, fontweight='bold')
    ax.set_title('DenseNet121 - Métricas de Desempenho', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

    # Adicionar linhas de referência
    ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Excelente (0.8)')
    ax.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Bom (0.6)')
    ax.axhline(y=0.4, color='red', linestyle='--', alpha=0.5, label='Regular (0.4)')
    ax.legend(loc='lower left')

    plt.tight_layout()
    plt.savefig(os.path.join(pasta_resultados, 'grafico_metricas.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Gráfico de métricas salvo em: grafico_metricas.png")
else:
    print("⚠️ Nenhuma métrica disponível para plotar.")

print("\n✅ Visualização completa dos resultados!")

## ConvNeXt V2

CNN moderna. Use este bloco para o terceiro experimento.

In [ ]:
# ============================================================
# ConvNeXt V2 + SVM — Extração de Features e Classificação
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import numpy as np
from datetime import datetime

import torch
import timm
from PIL import Image
from torchvision import transforms

from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    roc_auc_score,
    classification_report,
)
import joblib

# ============================================================
# 2. LIBERAÇÃO DE MEMÓRIA DE MODELOS ANTERIORES
# ============================================================

try:
    from tensorflow.keras import backend as K
    del model
    del model_resnet
    K.clear_session()
    gc.collect()
    print("✓ Memória dos modelos anteriores liberada")
except Exception:
    print("Nada para limpar")

# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def calcular_metricas_svm(y_true, y_pred, y_pred_proba=None, num_classes=None):
    """
    Calcula todas as métricas e retorna um dicionário com os resultados.
    Suporta decision_function como proxy para probabilidades quando necessário.
    """
    metricas = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Kappa":     cohen_kappa_score(y_true, y_pred),
    }

    if y_pred_proba is not None:
        if num_classes is None:
            num_classes = y_pred_proba.shape[1] if y_pred_proba.ndim > 1 else 2

        if num_classes == 2 or y_pred_proba.ndim == 1:
            scores = y_pred_proba if y_pred_proba.ndim == 1 else y_pred_proba[:, 1]
            metricas["AUC"] = roc_auc_score(y_true, scores)
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metricas["AUC"] = roc_auc_score(
                y_true_bin, y_pred_proba, multi_class="ovr", average="weighted"
            )
    else:
        metricas["AUC"] = None

    return metricas


def salvar_resultados_convnext(nome_modelo, metricas, y_true, y_pred,
                               y_pred_proba=None, diretorio="resultados",
                               info_extra=None):
    """
    Salva métricas, predições e relatório do modelo em arquivos locais.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pasta = os.path.join(diretorio, f"{nome_modelo}_{timestamp}")
    os.makedirs(pasta, exist_ok=True)

    # Serializa métricas para JSON
    metricas_serial = {
        k: (float(v) if hasattr(v, "item") else v)
        for k, v in metricas.items()
    }

    with open(os.path.join(pasta, "metricas.json"), "w") as f:
        json.dump(
            {"modelo": nome_modelo, "data_execucao": timestamp,
             "metricas": metricas_serial, "info_extra": info_extra},
            f, indent=4,
        )

    np.save(os.path.join(pasta, "y_true.npy"), y_true)
    np.save(os.path.join(pasta, "y_pred.npy"), y_pred)
    if y_pred_proba is not None:
        np.save(os.path.join(pasta, "y_pred_proba.npy"), y_pred_proba)

    report = classification_report(y_true, y_pred, output_dict=True)
    with open(os.path.join(pasta, "classification_report.json"), "w") as f:
        json.dump(report, f, indent=4)

    with open(os.path.join(pasta, "resumo.txt"), "w", encoding="utf-8") as f:
        f.write("=" * 50 + "\n")
        f.write(f"RESUMO DO MODELO: {nome_modelo}\n")
        f.write("=" * 50 + "\n")
        f.write(f"Data: {timestamp}\n\n")
        f.write("MÉTRICAS:\n" + "-" * 30 + "\n")
        for metrica, valor in metricas_serial.items():
            linha = f"{metrica}: {valor:.4f}\n" if valor is not None else f"{metrica}: Não disponível\n"
            f.write(linha)
        f.write("\nCONFIGURAÇÃO DO MODELO:\n" + "-" * 30 + "\n")
        f.write("Arquitetura: ConvNeXt V2 Tiny\n")
        f.write("Feature Extractor: timm (PyTorch)\n")
        f.write("Classificador: SVM RBF\n")
        f.write("Pré-processamento: StandardScaler\n")
        if info_extra:
            for k, v in info_extra.items():
                f.write(f"{k}: {v}\n")

    print(f"\n✓ Resultados salvos em: {pasta}")
    for arq in ["metricas.json", "y_true.npy", "y_pred.npy",
                "y_pred_proba.npy" if y_pred_proba is not None else None,
                "classification_report.json", "resumo.txt"]:
        if arq:
            print(f"  - {arq}")

    return pasta


def carregar_imagens_convnext(lista_paths, extractor, transform, device, batch_size=16):
    """
    Extrai features de uma lista de imagens em batches para economizar memória.
    """
    todas_features = []
    total = len(lista_paths)
    print(f"Processando {total} imagens em batches de {batch_size}...")

    for i in range(0, total, batch_size):
        batch_paths = lista_paths[i:i + batch_size]
        batch_tensor = torch.stack([
            transform(Image.open(p).convert("RGB"))
            for p in batch_paths
        ]).to(device)

        with torch.no_grad():
            features = extractor(batch_tensor).cpu().numpy()

        todas_features.append(features)

        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

        print(f"  ✓ Imagens {i + 1} a {min(i + batch_size, total)} processadas")

    return np.vstack(todas_features)

# ============================================================
# 4. CONFIGURAÇÃO DO MODELO
# ============================================================

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando device: {device}")

print("\nCarregando ConvNeXt V2 Tiny...")
feature_extractor_convnext = timm.create_model(
    "convnextv2_tiny", pretrained=True, num_classes=0
).to(device)
feature_extractor_convnext.eval()

with torch.no_grad():
    n_features = feature_extractor_convnext(torch.randn(1, 3, 224, 224).to(device)).shape[1]
print(f"✓ Features extraídas: {n_features} dimensões")

transform_convnext = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ============================================================
# 5. EXTRAÇÃO DE FEATURES
# ============================================================

print("\n" + "=" * 60)
print("EXTRAÇÃO DE FEATURES")
print("=" * 60)

print("\nCarregando imagens de treino...")
X_train_convnext = carregar_imagens_convnext(
    X_train_paths.tolist(), feature_extractor_convnext, transform_convnext, device
)
print(f"✓ X_train shape: {X_train_convnext.shape}")

print("\nCarregando imagens de teste...")
X_test_convnext = carregar_imagens_convnext(
    X_test_paths.tolist(), feature_extractor_convnext, transform_convnext, device
)
print(f"✓ X_test shape: {X_test_convnext.shape}")

print("\nAplicando StandardScaler...")
scaler_convnext = StandardScaler()
X_train_convnext = scaler_convnext.fit_transform(X_train_convnext)
X_test_convnext  = scaler_convnext.transform(X_test_convnext)
print("✓ Dados normalizados")

# ============================================================
# 6. TREINAMENTO SVM
# ============================================================

print("\n" + "=" * 60)
print("TREINAMENTO SVM")
print("=" * 60)

svm = SVC(
    kernel="rbf",
    C=0.5,
    gamma="scale",
    probability=True,
    random_state=42,
)
svm.fit(X_train_convnext, y_train)
print("✓ SVM treinado com sucesso!")

os.makedirs("models", exist_ok=True)
joblib.dump(svm, "models/convnextv2_svm.joblib")
joblib.dump(scaler_convnext, "models/convnextv2_scaler.joblib")
print("✓ Modelo SVM salvo em: models/convnextv2_svm.joblib")
print("✓ Scaler salvo em: models/convnextv2_scaler.joblib")

# ============================================================
# 7. AVALIAÇÃO DO MODELO
# ============================================================

print("\n" + "=" * 60)
print("AVALIAÇÃO DO MODELO")
print("=" * 60)

y_pred_convnext = svm.predict(X_test_convnext)

usar_decision_function = False
try:
    y_pred_proba_convnext = svm.predict_proba(X_test_convnext)
    print("✓ Usando predict_proba para AUC")
except Exception:
    try:
        y_pred_proba_convnext = svm.decision_function(X_test_convnext)
        usar_decision_function = True
        print("✓ Usando decision_function como proxy para AUC")
    except Exception:
        y_pred_proba_convnext = None
        print("⚠️ AUC não disponível")

metricas_convnext = calcular_metricas_svm(
    y_test,
    y_pred_convnext,
    y_pred_proba_convnext,
    num_classes=len(label_encoder.classes_),
)

print("\n" + "=" * 50)
print("MÉTRICAS — ConvNeXt V2 + SVM")
print("=" * 50)
for metrica, valor in metricas_convnext.items():
    if valor is not None:
        print(f"  {metrica:<12}: {valor:.4f}")
    else:
        print(f"  {metrica:<12}: Não disponível")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_convnext, target_names=label_encoder.classes_))

if usar_decision_function:
    print("\nNota: AUC calculada via decision_function. Para maior precisão, use probability=True no SVC.")

# ============================================================
# 8. VALIDAÇÃO CRUZADA
# ============================================================

print("\n" + "=" * 60)
print("VALIDAÇÃO CRUZADA — ConvNeXt V2 + SVM")
print("=" * 60)

cv_scores = cross_val_score(svm, X_train_convnext, y_train, cv=5)
print(f"CV Score médio : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"Scores por fold: {cv_scores}")
print(f"Acurácia no teste   : {metricas_convnext['Accuracy']:.4f}")
print(f"Diferença (teste-CV): {metricas_convnext['Accuracy'] - cv_scores.mean():.4f}")

# ============================================================
# 9. SALVAR RESULTADOS
# ============================================================

print("\n" + "=" * 60)
print("SALVANDO RESULTADOS")
print("=" * 60)

info_extra = {
    "feature_extractor": "ConvNeXt V2 Tiny",
    "n_features": int(n_features),
    "classificador": "SVM RBF",
    "kernel": "rbf",
    "C": 0.5,
    "gamma": "scale",
    "probability": True,
    "scaler": "StandardScaler",
    "device": device,
    "batch_size": 16,
}

pasta_resultados_convnext = salvar_resultados_convnext(
    nome_modelo="ConvNeXtV2_SVM",
    metricas=metricas_convnext,
    y_true=y_test,
    y_pred=y_pred_convnext,
    y_pred_proba=y_pred_proba_convnext,
    diretorio="resultados",
    info_extra=info_extra,
)

# ============================================================
# 10. RESUMO FINAL
# ============================================================

print(f"\n{'=' * 60}")
print("✅ RESUMO FINAL — ConvNeXt V2 + SVM")
print(f"{'=' * 60}")
print(f"✓ Acurácia : {metricas_convnext['Accuracy']:.4f}")
print(f"✓ F1-Score : {metricas_convnext['F1-Score']:.4f}")
print(f"✓ AUC      : {metricas_convnext['AUC']:.4f}" if metricas_convnext["AUC"] else "✓ AUC: Não disponível")
print(f"✓ Kappa    : {metricas_convnext['Kappa']:.4f}")
print(f"✓ Features : {n_features}")
print(f"✓ Device   : {device}")
print(f"✓ Pasta    : {pasta_resultados_convnext}")

# ============================================================
# 11. LIBERAÇÃO DE MEMÓRIA GPU
# ============================================================

if device == "cuda":
    del feature_extractor_convnext
    torch.cuda.empty_cache()
    gc.collect()
    print("\n✓ Memória GPU liberada")

In [ ]:
# ============================================
# VISUALIZAÇÕES DOS RESULTADOS
# ============================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle

# Configuração de estilo
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 12

# Nomes das classes
class_names = label_encoder.classes_
n_classes = len(class_names)

print("\n" + "="*60)
print("GERANDO VISUALIZAÇÕES")
print("="*60)

# ------------------------------------------------------------
# 1. MATRIZ DE CONFUSÃO
# ------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred_convnext)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Contagem'})
plt.title('Matriz de Confusão - ConvNeXt V2 + SVM', fontsize=14, pad=15)
plt.xlabel('Predito', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(pasta_resultados_convnext, 'matriz_confusao.png'), bbox_inches='tight')
plt.show()
print("✓ Matriz de confusão salva e exibida")

# ------------------------------------------------------------
# 2. TABELA DE MÉTRICAS
# ------------------------------------------------------------
# DataFrame com as métricas
df_metrics = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Kappa', 'AUC'],
    'Valor': [
        metricas_convnext['Accuracy'],
        metricas_convnext['Precision'],
        metricas_convnext['Recall'],
        metricas_convnext['F1-Score'],
        metricas_convnext['Kappa'],
        metricas_convnext['AUC'] if metricas_convnext['AUC'] is not None else np.nan
    ]
})
df_metrics['Valor'] = df_metrics['Valor'].apply(lambda x: f"{x:.4f}" if not np.isnan(x) else "N/A")

# Exibir tabela formatada
print("\nTabela de Métricas:")
print("="*30)
print(df_metrics.to_string(index=False))
print("="*30)

# Salvar como CSV
df_metrics.to_csv(os.path.join(pasta_resultados_convnext, 'tabela_metricas.csv'), index=False)
print("✓ Tabela de métricas salva como CSV")

# ------------------------------------------------------------
# 3. GRÁFICO DE BARRAS DAS MÉTRICAS
# ------------------------------------------------------------
# Preparar dados (apenas métricas disponíveis)
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Kappa']
metric_values = [metricas_convnext[name] for name in metric_names]
if metricas_convnext['AUC'] is not None:
    metric_names.append('AUC')
    metric_values.append(metricas_convnext['AUC'])

colors = sns.color_palette("viridis", len(metric_names))

plt.figure(figsize=(10, 6))
bars = plt.bar(metric_names, metric_values, color=colors, edgecolor='black', linewidth=0.5)

# Adicionar valores sobre as barras
for bar, val in zip(bars, metric_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.ylim(0, 1.1)
plt.title('Métricas de Desempenho - ConvNeXt V2 + SVM', fontsize=14, pad=15)
plt.ylabel('Valor')
plt.xlabel('Métrica')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(pasta_resultados_convnext, 'barras_metricas.png'), bbox_inches='tight')
plt.show()
print("✓ Gráfico de barras salvo e exibido")

# ------------------------------------------------------------
# 4. CURVA ROC (MULTICLASSE - ONE-VS-REST)
# ------------------------------------------------------------
if y_pred_proba_convnext is not None and not usar_decision_function:
    # Binarizar os labels
    y_test_bin = label_binarize(y_test, classes=range(n_classes))

    # Calcular curvas ROC para cada classe
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    plt.figure(figsize=(10, 8))
    colors_roc = cycle(sns.color_palette("bright", n_classes))

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_proba_convnext[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], color=next(colors_roc), lw=2,
                 label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')

    # Linha de referência (classificador aleatório)
    plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.7, label='Aleatório (AUC = 0.500)')

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Taxa de Falsos Positivos (1 - Especificidade)', fontsize=12)
    plt.ylabel('Taxa de Verdadeiros Positivos (Sensibilidade)', fontsize=12)
    plt.title('Curvas ROC (One-vs-Rest) - ConvNeXt V2 + SVM', fontsize=14, pad=15)
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(pasta_resultados_convnext, 'curva_roc.png'), bbox_inches='tight')
    plt.show()
    print("✓ Curvas ROC salvas e exibidas")
elif usar_decision_function:
    print("\n⚠️ Curva ROC não gerada: usando decision_function, não probabilidades.")
    print("   Para curvas ROC precisas, garanta probability=True no SVC.")
else:
    print("\n⚠️ Curva ROC não disponível (sem scores de predição).")

print("\n" + "="*60)
print("✅ TODAS AS VISUALIZAÇÕES FORAM GERADAS COM SUCESSO!")
print(f"   Arquivos salvos em: {pasta_resultados_convnext}")
print("="*60)

In [ ]:
# Comparação resumida

# Carregar os arrays salvos se as variáveis não estiverem na memória
import numpy as np
import glob

def carregar_ultima_predicao(nome_modelo):
    """Carrega a última predição salva para um modelo"""
    pasta = f"resultados/{nome_modelo}_*"
    pastas = sorted(glob.glob(pasta))
    if not pastas:
        return None
    ultima = pastas[-1]
    y_pred = np.load(f"{ultima}/y_pred.npy")
    return y_pred

# Tenta carregar do ambiente ou dos arquivos salvos
try:
    y_pred_densenet = y_pred  # do bloco DenseNet121
except NameError:
    y_pred_densenet = carregar_ultima_predicao("DenseNet121")

try:
    y_pred_resnet = y_pred_resnet  # do bloco ResNet50
except NameError:
    y_pred_resnet = carregar_ultima_predicao("ResNet50")

# ConvNeXt V2 já está na memória como y_pred_convnext
try:
    y_pred_convnext = y_pred_convnext
except NameError:
    y_pred_convnext = carregar_ultima_predicao("ConvNextV2_SVM")

# Montar dicionário com apenas os que foram carregados
resultados_carregados = {}
if y_pred_densenet is not None:
    resultados_carregados["DenseNet121"] = accuracy_score(y_test, y_pred_densenet)
if y_pred_resnet is not None:
    resultados_carregados["ResNet50"] = accuracy_score(y_test, y_pred_resnet)
if y_pred_convnext is not None:
    resultados_carregados["ConvNeXt V2"] = accuracy_score(y_test, y_pred_convnext)

print("\n=== RESUMO DE RESULTADOS ===")
for modelo, acc in resultados_carregados.items():
    print(f"{modelo:20s}: {acc:.4f}")